# Safety Monitor Evaluation


## Setup

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

os.chdir('/Users/diba/Desktop/Academic/Projects/RealTimeSafetyMonitor')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## Load Model and Classifier

In [ ]:
# Load BLIP
model_name = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name).to(device)
model.eval()

# Safety Classifier - Assumes hazard unless clearly safe
class SafetyClassifier:
    def __init__(self):
        
        self.hazard_keywords = [
            # Electrical
            'wire', 'cable', 'cord', 'plug', 'socket', 'outlet', 'electrical', 'power',
            'extension', 'charger', 'adapter', 'electric',
            # Objects on floor/surfaces (potential trip hazards)
            'floor', 'ground', 'lying', 'scattered', 'pile', 'stack', 'stacked',
            # Clutter/mess
            'mess', 'cluttered', 'clutter', 'disorganized', 'items', 'objects',
            'papers', 'files', 'stuff',
            # Liquids
            'wet', 'water', 'liquid', 'spill', 'puddle', 'moisture', 'damp',
            # Damage
            'broken', 'damaged', 'cracked', 'shattered', 'glass', 'sharp',
            # Obstructions
            'blocked', 'obstruct', 'narrow', 'crowded', 'packed',
            # Fire/heat
            'fire', 'flame', 'smoke', 'burning', 'hot', 'burn',
            # Generic objects that could be hazards
            'bag', 'backpack', 'box', 'boxes', 'equipment', 'tool', 'tools',
            'furniture', 'drawer', 'open', 'edge'
        ]
        
        # Very clear and strong safe indicators only
        self.safe_keywords = [
            'clean', 'organized', 'tidy', 'clear', 'empty', 'neat', 'spacious',
            'well-lit', 'bright', 'orderly', 'maintained', 'unobstructed',
            'organized', 'polished', 'pristine'
        ]
        
        # Words that indicate a normal, safe scene
        self.neutral_safe_words = [
            'hallway', 'corridor', 'room', 'wall', 'ceiling', 'door',
            'window', 'light', 'building'
        ]
    
    def classify(self, caption):
        caption_lower = caption.lower()
        
        # Strong safe indicators AND no hazard keywords → SAFE
        has_safe_keyword = any(keyword in caption_lower for keyword in self.safe_keywords)
        has_hazard_keyword = any(keyword in caption_lower for keyword in self.hazard_keywords)
        
        if has_safe_keyword and not has_hazard_keyword:
            return 'safe'
        
        # Checking ANY hazard keyword → HAZARD
        if has_hazard_keyword:
            return 'hazard'
        
        # Cgecking Only neutral architectural words and no hazards → SAFE
        has_neutral = any(keyword in caption_lower for keyword in self.neutral_safe_words)
        if has_neutral and not has_hazard_keyword:
            # Check if caption is very short and generic (likely a safe empty space)
            if len(caption_lower.split()) <= 6:
                return 'safe'
        
        # Default: Conservative approach - assume HAZARD for safety
    
        return 'hazard'

classifier = SafetyClassifier()
print("Model and AGGRESSIVE classifier loaded")
print("Strategy: Conservative - assumes hazard unless clearly safe")

## Loading Test Data

In [ ]:
# Load annotations
annotations_path = 'data/training/annotations2.csv'
df = pd.read_csv(annotations_path)

# Create binary labels from has_hazard column
def convert_to_binary(has_hazard_value):
    if pd.isna(has_hazard_value):
        return 'unknown'
    val = str(has_hazard_value).lower().strip()
    if val in ['yes', 'y', 'true', '1']:
        return 'hazard'
    elif val in ['no', 'n', 'false', '0']:
        return 'safe'
    else:
        return 'unknown'

df['ground_truth'] = df['has_hazard'].apply(convert_to_binary)

# Filter out unknown labels
df_eval = df[df['ground_truth'] != 'unknown'].copy()

print(f"Total test samples: {len(df_eval)}")
print(f"\nClass distribution:")
print(df_eval['ground_truth'].value_counts())

## Evaluation

In [ ]:
def generate_caption(image_path):
    """Generate caption for image"""
    image = Image.open(image_path).convert('RGB')
    inputs = processor(image, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=30)
    
    return processor.decode(output_ids[0], skip_special_tokens=True)

# Evaluate all images
results = []
img_dir = 'data/training/images'

print("Evaluating system...\n")

for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
    img_path = os.path.join(img_dir, row['filename'])
    
    if not os.path.exists(img_path):
        continue
    
    # Generate caption and classify
    caption = generate_caption(img_path)
    prediction = classifier.classify(caption)
    
    results.append({
        'filename': row['filename'],
        'ground_truth': row['ground_truth'],
        'prediction': prediction,
        'caption': caption,
        'correct': row['ground_truth'] == prediction
    })

results_df = pd.DataFrame(results)
print(f"\nEvaluation complete! Processed {len(results_df)} images")

## Calculate Metrics

In [ ]:
# Overall accuracy
accuracy = results_df['correct'].mean()

# Get predictions and ground truth
y_true = results_df['ground_truth'].values
y_pred = results_df['prediction'].values

# Get unique classes actually present
unique_classes = sorted(list(set(y_true) | set(y_pred)))

# Classification report (only for classes that exist)
print("="*70)
print("PERFORMANCE METRICS")
print("="*70)
print(f"\nOverall Accuracy: {accuracy*100:.1f}%")
print(f"\nClassification Report:\n")
print(classification_report(y_true, y_pred, labels=unique_classes, target_names=unique_classes, zero_division=0))

# Confusion matrix (only for classes that exist)
cm = confusion_matrix(y_true, y_pred, labels=unique_classes)
print("\nConfusion Matrix:")
print(f"Labels: {unique_classes}")
print(cm)

## Visualize Results

In [ ]:
# Get unique classes for visualization
unique_classes = ['hazard', 'safe']

# Confusion Matrix Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=unique_classes,
            yticklabels=unique_classes,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Safety Detection System\n(68.6% Accuracy, Conservative Approach)', 
          fontsize=12, fontweight='bold')
plt.ylabel('Ground Truth', fontsize=11)
plt.xlabel('Predicted', fontsize=11)

# Add percentage annotations
for i in range(len(unique_classes)):
    for j in range(len(unique_classes)):
        total = cm[i].sum()
        percentage = cm[i, j] / total * 100 if total > 0 else 0
        plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Confusion matrix saved as confusion_matrix.png")

## Error Analysis

In [ ]:
# Show misclassified examples
errors = results_df[~results_df['correct']]

print(f"\nMisclassified Examples ({len(errors)} total):\n")
print("="*100)

for idx, row in errors.head(10).iterrows():
    print(f"\nFile: {row['filename']}")
    print(f"Ground Truth: {row['ground_truth']}")
    print(f"Predicted: {row['prediction']}")
    print(f"Caption: {row['caption']}")
    print("-" * 100)

## Save Results

In [ ]:
# Calculate metrics first
from sklearn.metrics import precision_recall_fscore_support

# Overall weighted metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=['hazard', 'safe'], average='weighted', zero_division=0
)

# Per-class metrics
precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
    y_true, y_pred, labels=['hazard', 'safe'], zero_division=0
)

# Save detailed results
results_df.to_csv('evaluation_results.csv', index=False)

# Save enhanced summary report
with open('evaluation_report.txt', 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("SAFETY MONITORING SYSTEM - EVALUATION REPORT\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("MODEL CONFIGURATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Vision Model: BLIP Baseline (Salesforce/blip-image-captioning-base)\n")
    f.write(f"Classification: Keyword-based with aggressive hazard detection\n")
    f.write(f"Strategy: Conservative - prioritizes catching hazards over avoiding false alarms\n\n")
    
    f.write("DATASET\n")
    f.write("-" * 80 + "\n")
    f.write(f"Total test samples: {len(results_df)}\n")
    f.write(f"Hazard samples: {support_per_class[0]}\n")
    f.write(f"Safe samples: {support_per_class[1]}\n\n")
    
    f.write("OVERALL PERFORMANCE\n")
    f.write("-" * 80 + "\n")
    f.write(f"Accuracy: {accuracy*100:.1f}%\n")
    f.write(f"Weighted F1 Score: {f1*100:.1f}%\n\n")
    
    f.write("HAZARD DETECTION (Primary Goal)\n")
    f.write("-" * 80 + "\n")
    f.write(f"Precision: {precision_per_class[0]*100:.1f}%\n")
    f.write(f"Recall: {recall_per_class[0]*100:.1f}% <- Catches {recall_per_class[0]*100:.0f}% of hazards!\n")
    f.write(f"F1 Score: {f1_per_class[0]*100:.1f}%\n")
    f.write(f"Missed hazards: {support_per_class[0] - cm[0][0]} out of {support_per_class[0]}\n\n")
    
    f.write("SAFE SCENE DETECTION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Precision: {precision_per_class[1]*100:.1f}%\n")
    f.write(f"Recall: {recall_per_class[1]*100:.1f}%\n")
    f.write(f"F1 Score: {f1_per_class[1]*100:.1f}%\n")
    f.write(f"False alarms (safe marked as hazard): {cm[1][0]} out of {support_per_class[1]}\n\n")
    
    f.write("CONFUSION MATRIX\n")
    f.write("-" * 80 + "\n")
    f.write(f"                Predicted Hazard    Predicted Safe\n")
    f.write(f"Actual Hazard   {cm[0][0]:^16}    {cm[0][1]:^14}\n")
    f.write(f"Actual Safe     {cm[1][0]:^16}    {cm[1][1]:^14}\n\n")
    
    f.write("INTERPRETATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"The system prioritizes safety by using a conservative classification approach.\n")
    f.write(f"It successfully detects {recall_per_class[0]*100:.0f}% of hazards, with {cm[1][0]} false alarms\n")
    f.write(f"on safe scenes. This trade-off favors catching real hazards over avoiding\n")
    f.write(f"false positives, which is appropriate for a safety monitoring application.\n\n")
    
    f.write("RECOMMENDATIONS FOR IMPROVEMENT\n")
    f.write("-" * 80 + "\n")
    f.write("1. Collect 500-1000 annotated safety images for fine-tuning\n")
    f.write("2. Use domain-specific vision-language model (e.g., CLIP with safety concepts)\n")
    f.write("3. Implement temporal filtering for video streams (reduce false alarm rate)\n")
    f.write("4. Add confidence scores to allow users to adjust sensitivity\n")
    f.write("5. Incorporate object detection for specific hazard types (cables, spills, etc.)\n")

print("\n Results saved:")
print("  - evaluation_results.csv (detailed per-image results)")
print("  - evaluation_report.txt (professional summary)")
print("  - confusion_matrix.png (visualization)")

## Performance Summary

In [ ]:
# Calculate final metrics
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=['hazard', 'safe'], average='weighted', zero_division=0
)

print("\n" + "=" * 80)
print("FINAL PERFORMANCE SUMMARY")
print("=" * 80)
print(f"\n{'Metric':<30} {'Value':>15}")
print("-" * 80)
print(f"{'Overall Accuracy':<30} {accuracy*100:>14.1f}%")
print(f"{'Weighted Precision':<30} {precision*100:>14.1f}%")
print(f"{'Weighted Recall':<30} {recall*100:>14.1f}%")
print(f"{'Weighted F1 Score':<30} {f1*100:>14.1f}%")
print()
print(f"{'Total Test Images':<30} {len(results_df):>15}")
print(f"{'Correct Predictions':<30} {results_df['correct'].sum():>15}")
print(f"{'Incorrect Predictions':<30} {len(results_df) - results_df['correct'].sum():>15}")

# Detailed breakdown
print("\n" + "=" * 80)
print("PER-CLASS PERFORMANCE")
print("=" * 80)

for idx, label in enumerate(['hazard', 'safe']):
    prec_class, rec_class, f1_class, supp_class = precision_recall_fscore_support(
        y_true, y_pred, labels=[label], zero_division=0
    )
    
    print(f"\n{label.upper()} DETECTION:")
    print(f"  Precision: {prec_class[0]*100:>6.1f}%")
    print(f"  Recall:    {rec_class[0]*100:>6.1f}%")
    print(f"  F1 Score:  {f1_class[0]*100:>6.1f}%")
    print(f"  Support:   {supp_class[0]:>6} samples")

print("\n" + "=" * 80)
print("KEY INSIGHT FOR SAFETY SYSTEMS")
print("=" * 80)
print(f"✓ Hazard Detection Recall: {recall_per_class[0]*100:.0f}% - Catches most real hazards!")
print(f"⚠ False Alarm Rate: {cm[1][0]} safe scenes flagged as hazards")
print(f"→ Trade-off: Conservative approach prioritizes safety over precision")
print("=" * 80)